<img src="../../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# pandas for Exploratory Data Analysis - imdb data - Solution

---

In [ ]:
'''
BASIC LEVEL
'''

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# read in 'imdb_1000.csv' and store it in a DataFrame named movies
imdb_1000_data_url = '../data/imdb_1000.csv'
movies = pd.read_csv(imdb_1000_data_url)

In [ ]:
# check the number of rows and columns
movies.shape

In [ ]:
# check the data type of each column
movies.dtypes

In [ ]:
# calculate the average movie duration
movies.duration.mean()

In [ ]:
# sort the DataFrame by duration to find the shortest and longest movies
movies.sort_values('duration').head(1)
movies.sort_values('duration').tail(1)

In [ ]:
# create a histogram of duration, choosing an "appropriate" number of bins
movies.duration.plot(kind='hist', bins=20)

In [ ]:
# use a box plot to display that same data
movies.duration.plot(kind='box')

In [ ]:
'''
INTERMEDIATE LEVEL
'''

In [ ]:
# count how many movies have each of the content ratings
movies.content_rating.value_counts()

In [ ]:
# use a visualization to display that same data, including a title and x and y labels
movies.content_rating.value_counts().plot(kind='bar', title='Top 1000 Movies by Content Rating')
plt.xlabel('Content Rating')
plt.ylabel('Number of Movies')

In [ ]:
# convert the following content ratings to "UNRATED": NOT RATED, APPROVED, PASSED, GP
movies.content_rating.replace(['NOT RATED', 'APPROVED', 'PASSED', 'GP'], 'UNRATED', inplace=True)

In [ ]:
# convert the following content ratings to "NC-17": X, TV-MA
movies.content_rating.replace(['X', 'TV-MA'], 'NC-17', inplace=True)

In [ ]:
# count the number of missing values in each column
movies.isnull().sum()

In [ ]:
# if there are missing values: examine them, then fill them in with "reasonable" values
movies[movies.content_rating.isnull()]
movies.content_rating.fillna('UNRATED', inplace=True)

In [ ]:
# calculate the average star rating for movies 2 hours or longer,
# and compare that with the average star rating for movies shorter than 2 hours
movies[movies.duration >= 120].star_rating.mean()
movies[movies.duration < 120].star_rating.mean()

In [ ]:
# use a visualization to detect whether there is a relationship between duration and star rating
movies.plot(kind='scatter', x='duration', y='star_rating', alpha=0.2)

In [ ]:
# calculate the average duration for each genre
movies.groupby('genre').duration.mean()

In [ ]:
'''
ADVANCED LEVEL
'''

In [ ]:
# visualize the relationship between content rating and duration
movies.boxplot(column='duration', by='content_rating')
movies.hist(column='duration', by='content_rating', sharex=True)

In [ ]:
# determine the top rated movie (by star rating) for each genre
movies.sort_values('star_rating', ascending=False).groupby('genre').title.first()
movies.groupby('genre').title.first()   # equivalent, since DataFrame is already sorted by star rating

In [ ]:
# check if there are multiple movies with the same title, and if so, determine if they are actually duplicates
dupe_titles = movies[movies.title.duplicated()].title
movies[movies.title.isin(dupe_titles)]

In [ ]:
# calculate the average star rating for each genre, but only include genres with at least 10 movies
genre_counts=movies.groupby('genre')['genre'].count()
big_genre=list(genre_counts[genre_counts>10].index)
movies.loc[movies['genre'].isin(big_genre)].groupby('genre')['star_rating'].mean().sort_values()

In [ ]:
# option 1: manually create a list of relevant genres, then filter using that list
movies.genre.value_counts()
top_genres = ['Drama', 'Comedy', 'Action', 'Crime', 'Biography', 'Adventure', 'Animation', 'Horror', 'Mystery']
movies[movies.genre.isin(top_genres)].groupby('genre').star_rating.mean()

In [ ]:
# option 2: automatically create a list of relevant genres by saving the value_counts and then filtering
genre_counts = movies.genre.value_counts()
top_genres = genre_counts[genre_counts >= 10].index
movies[movies.genre.isin(top_genres)].groupby('genre').star_rating.mean()

In [ ]:
# option 3: calculate the average star rating for all genres, then filter using a boolean Series
movies.groupby('genre').star_rating.mean()[movies.genre.value_counts() >= 10]

In [ ]:
# option 4: aggregate by count and mean, then filter using the count
genre_ratings = movies.groupby('genre').star_rating.agg(['count', 'mean'])
genre_ratings[genre_ratings['count'] >= 10]

In [ ]:
'''
BONUS
'''

#### Figure out something "interesting" using the actors data!

In [ ]:
# Let's display a list of movies that include Marlon Brando.
movies.loc[movies['actors_list'].str.contains('Marlon Brando')]

In [ ]:
# How many movies is Samuel Jackson in?
len(movies.loc[movies['actors_list'].str.contains('Samuel L. Jackson')])

In [ ]:
# Let's explore a little further.
print(movies.iloc[0]['actors_list'])
# That 'u' is part of the external representation of the string, 
# meaning it's a Unicode string as opposed to a byte string. 

# Let's take a look at the first row of the dataframe, and break up that string of actors into its parts.
line = movies.iloc[0]['actors_list']
import re
line = re.sub("u'", "'", line)
line = re.sub("'", "", line)
line = re.sub('\[', "", line)
line = re.sub("\]", "", line)
actors_list=line.split(', ')
print(actors_list)
print(len(actors_list))
print(actors_list[0])

In [ ]:
# Now let's write a function that displays all the actors in a clean list.
import re
def actors_cleanlist(row):
    line = row['actors_list']   
    line = re.sub("u'", "'", line)
    line = re.sub("'", "", line)
    line = re.sub('\[', "", line)
    line = re.sub("\]", "", line)
    actors_list=line.split(', ')
    return(actors_list)
actors_cleanlist(movies.iloc[0])

In [ ]:
# Let's create a new column with that list.
movies['cleanlist']=movies.apply(actors_cleanlist, axis=1)
movies.head()

In [ ]:
# Okay, let's create columns for actors 1, 2, and 3
movies['actor_0']=movies['cleanlist'].apply(lambda row: row[0])
movies['actor_1']=movies['cleanlist'].apply(lambda row: row[1])
movies['actor_2']=movies['cleanlist'].apply(lambda row: row[2])
movies.head()

In [ ]:
# Okay, who's the most frequent lead actor?
movies['actor_0'].value_counts().head()

In [ ]:
# Who are the most frequent supporting actors?
print(movies['actor_1'].value_counts().head())
print('\n')
print(movies['actor_2'].value_counts().head())